# Library notebook — Post-processing and metrics

**Type:** library notebook (import via `%run`; not an experiment entry point).

## Purpose

Morphological post-processing (`pos_process`), segmentation metrics (`calculate_metrics`), identifier-based label pairing, and path validation.

**Consumers:** `fetal_vein_segmentation.ipynb`, `evaluation.ipynb`.

## Usage notes

Import with `%run`. Validation cells are optional when used as a library.


## Dependencies

In [ ]:
%matplotlib inline

import re
from pathlib import Path

import numpy as np
import matplotlib.image as mpimg
from scipy.ndimage import binary_opening, label

## Function definitions — project paths

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """Walk up the directory tree until 02_dataset/ is found."""
    for candidate_path in [start_path, *start_path.parents]:
        if (candidate_path / "02_dataset").is_dir():
            return candidate_path
    raise FileNotFoundError(
        f"Directory 02_dataset/ not found from {start_path}. "
        "Run the notebook from the fetal_vein_segmentation repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "02_dataset"
IMAGES_ORIGINAL_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
SAVE_MODELS_DIR = DATA_DIR / "Save_Models"

PREPROCESSED_IMAGE_DIRS = [
    DATA_DIR / "images_pp_1",
    DATA_DIR / "images_pp_2",
    DATA_DIR / "images_pp_3",
    DATA_DIR / "images_pp_4",
    DATA_DIR / "images_pp_5",
]

RESULT_FOLDER_CONFIG = [
    ("Original", DATA_DIR / "results_original"),
    ("PP1", DATA_DIR / "results_pp_1"),
    ("PP2", DATA_DIR / "results_pp_2"),
    ("PP3", DATA_DIR / "results_pp_3"),
    ("PP4", DATA_DIR / "results_pp_4"),
    ("PP5", DATA_DIR / "results_pp_5"),
]

EXPECTED_MODEL_FILES = [
    "best_metric_model_original.pth",
    "best_metric_model_pp_1.pth",
    "best_metric_model_pp_2.pth",
    "best_metric_model_pp_3.pth",
    "best_metric_model_pp_4.pth",
    "best_metric_model_pp_5.pth",
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## Base identifier and pairing (never by list index)

In [ ]:
PP_SUFFIX_PATTERN = re.compile(r"_PP_PL_\d+$", re.IGNORECASE)
PATIENT_ID_PATTERN = re.compile(r"^P\d+", re.IGNORECASE)


def extract_base_identifier(file_name: str) -> str:
    """
    Original image identifier (without preprocessing suffix).
    Ex.: P080_IMG1_PP_PL_1.png → P080_IMG1; P080_IMG1.png → P080_IMG1.
    """
    stem = Path(file_name).stem
    original_identifier = PP_SUFFIX_PATTERN.sub("", stem)
    if not original_identifier or not PATIENT_ID_PATTERN.match(original_identifier):
        raise ValueError(
            f"Invalid original identifier in '{file_name}'. "
            f"Expected format P080_IMG1 or P080_IMG1_PP_PL_N."
        )
    return original_identifier


def resolve_label_path(prediction_file_name: str, labels_dir: Path = LABELS_DIR) -> Path:
    """Maps image/prediction to ground truth in labels/ using the original identifier."""
    original_identifier = extract_base_identifier(prediction_file_name)
    candidate_path = labels_dir / f"{original_identifier}.png"
    if candidate_path.is_file():
        return candidate_path
    raise FileNotFoundError(
        f"Missing label for '{prediction_file_name}' (ID original {original_identifier}). "
        f"Expected: {candidate_path}"
    )


def list_predictions(results_folder: Path) -> list:
    """Lists prediction files sorted by name (does not define pairing)."""
    if not results_folder.is_dir():
        return []
    return sorted(
        p
        for p in results_folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

## Path, dataset, and pairing validation

In [ ]:
def validate_project_paths() -> list:
    """Checks required folders; returns a list of warning strings."""
    warnings = []
    required_folders = [
        ("images (original)", IMAGES_ORIGINAL_DIR),
        ("labels", LABELS_DIR),
        ("Save_Models", SAVE_MODELS_DIR),
    ]
    for folder_name, folder_path in required_folders:
        if not folder_path.is_dir():
            warnings.append(f"Missing folder: {folder_name} → {folder_path}")

    for pipeline_index, folder_path in enumerate(PREPROCESSED_IMAGE_DIRS, start=1):
        if not folder_path.is_dir():
            warnings.append(f"Missing preprocessed dataset: images_pp_{pipeline_index}")
        elif len(list(folder_path.glob("*.png"))) == 0:
            warnings.append(f"images_pp_{pipeline_index} exists but contains no .png files")

    for experiment_key, results_folder in RESULT_FOLDER_CONFIG:
        if not results_folder.is_dir():
            warnings.append(f"Missing results folder: {experiment_key} → {results_folder.name}")
        elif len(list_predictions(results_folder)) == 0:
            warnings.append(f"{results_folder.name} exists but contains no predictions")

    for model_filename in EXPECTED_MODEL_FILES:
        model_path = SAVE_MODELS_DIR / model_filename
        if not model_path.is_file():
            warnings.append(f"Missing model: {model_filename}")

    return warnings


def validate_pairing_folder(images_folder: Path, labels_dir: Path = LABELS_DIR) -> list:
    """Validates each image/prediction → label by ID (not by index)."""
    pairing_errors = []
    if not images_folder.is_dir():
        return [f"Folder does not exist: {images_folder}"]
    for image_path in sorted(images_folder.glob("*.png")):
        try:
            resolve_label_path(image_path.name, labels_dir)
        except (ValueError, FileNotFoundError) as exc:
            pairing_errors.append(f"{image_path.name}: {exc}")
    return pairing_errors


def validate_all_pairings() -> dict:
    """Validates images, images_pp_*, and results folders."""
    pairing_report = {}
    pairing_report["images"] = validate_pairing_folder(IMAGES_ORIGINAL_DIR)
    for pipeline_index, folder_path in enumerate(PREPROCESSED_IMAGE_DIRS, start=1):
        pairing_report[f"images_pp_{pipeline_index}"] = validate_pairing_folder(folder_path)
    for experiment_key, results_folder in RESULT_FOLDER_CONFIG:
        pairing_errors = []
        for prediction_file in list_predictions(results_folder):
            try:
                resolve_label_path(prediction_file.name)
            except (ValueError, FileNotFoundError) as exc:
                pairing_errors.append(f"{prediction_file.name}: {exc}")
        pairing_report[f"results_{experiment_key}"] = pairing_errors
    return pairing_report


## Loading and orientation helpers

The lecturer notebook applies orientation correction before metrics (`Pos_Metrics.ipynb`).

In [ ]:
def load_mask_png(file_path: Path) -> np.ndarray:
    """Loads a PNG mask as a 2D array."""
    loaded_image = mpimg.imread(file_path)
    if loaded_image.ndim == 3:
        loaded_image = loaded_image[:, :, 0]
    return loaded_image


def binarize_mask(input_mask: np.ndarray, threshold: float = 127.0) -> np.ndarray:
    """Converts mask to binary {0, 1} uint8."""
    if input_mask.dtype == np.bool_:
        return input_mask.astype(np.uint8)
    if input_mask.max() <= 1.0:
        return (input_mask > 0.5).astype(np.uint8)
    return (input_mask >= threshold).astype(np.uint8)


def correct_prediction_orientation(prediction_array: np.ndarray) -> np.ndarray:
    """
    Corrects prediction orientation (equivalent to Pos_Metrics.ipynb).
    predicted = np.flip(np.rot90(predicted, 1), 0)
    """
    return np.flip(np.rot90(prediction_array, 1), 0)


## `pos_process()` — morphological post-processing

Lecturer skeleton: **opening** + **labeling** (largest connected component).

In [ ]:
def pos_process(predicted: np.ndarray, opening_kernel_size: int = 3) -> np.ndarray:
    """
    Post-processing: morphological opening + maior componente ligado.

    Args:
        predicted: binary or grayscale mask.
        opening_kernel_size: lado do elemento estruturante (quadrado).

    Returns:
        Binary uint8 mask {0, 1}.
    """
    mask_binary = binarize_mask(predicted)
    structuring_element = np.ones((opening_kernel_size, opening_kernel_size), dtype=bool)
    opened_mask = binary_opening(input_mask > 0, structure=structuring_element)

    labeled_mask, component_count = label(aberta)
    if component_count == 0:
        return np.zeros_like(mask_binary, dtype=np.uint8)

    largest_component_label = 1
    largest_component_area = 0
    for component_label in range(1, numero + 1):
        component_area = np.sum(labeled_mask == component_label)
        if area > largest_component_area:
            largest_component_area = area
            largest_component_label = rotulo

    predicted_pos_proc = (rotulos == largest_component_label).astype(np.uint8)
    return postprocessed_mask

## `calculate_metrics()` — Dice, Accuracy, Precision, Recall

In [ ]:
def calculate_metrics(predicted: np.ndarray, gt: np.ndarray):
    """
    Computes binary segmentation metrics.

    Returns:
        (dice, accuracy, precision, recall)
    """
    pred = binarize_mask(predicted)
    gt_bin = binarize_mask(gt)

    if pred.shape != gt_bin.shape:
        raise ValueError(
            f"Incompatible shapes: prediction {pred.shape} vs ground truth {gt_bin.shape}"
        )

    tp = int(np.sum((pred == 1) & (gt_bin == 1)))
    tn = int(np.sum((pred == 0) & (gt_bin == 0)))
    fp = int(np.sum((pred == 1) & (gt_bin == 0)))
    fn = int(np.sum((pred == 0) & (gt_bin == 1)))

    eps = 1e-8
    dice = (2.0 * tp) / (2.0 * tp + fp + fn + eps)
    ac = (tp + tn) / (tp + tn + fp + fn + eps)
    pr = tp / (tp + fp + eps)
    re = tp / (tp + fn + eps)

    return dice, ac, pr, re


def compute_mean_metrics(metric_tuples: list) -> dict:
    """Mean of (dice, accuracy, precision, recall) tuples."""
    if len(metric_tuples) == 0:
        return {"dice": np.nan, "accuracy": np.nan, "precision": np.nan, "recall": np.nan}
    arr = np.array(metric_tuples)
    return {
        "dice": float(np.mean(arr[:, 0])),
        "accuracy": float(np.mean(arr[:, 1])),
        "precision": float(np.mean(arr[:, 2])),
        "recall": float(np.mean(arr[:, 3])),
    }

## Quick validation (run after importing the library)

In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print("--- Path and dataset warnings ---")
for aviso in validate_project_paths():
    print(f"  [WARNING] {aviso}")
if not validate_project_paths():
    print("  Folder structure OK (or no critical warnings listed).")

print("--- Pairing images_pp_1 (example) ---")
pp1_pairing_errors = validate_pairing_folder(PREPROCESSED_IMAGE_DIRS[0])
if pp1_pairing_errors:
    for e in pp1_pairing_errors[:5]:
        print(f"  [ERROR] {e}")
    if len(pp1_pairing_errors) > 5:
        print(f"  … and {len(pp1_pairing_errors) - 5} more errors")
else:
    print("  images_pp_1: ID pairing OK (or folder empty/missing)")
